# Notebook 9: Decision Boundary Visualization

**Difficulty:** ⭐⭐⭐ | **Estimated time:** 2.5 hours | **Week 6 — Classification: Logistic Regression & Decision Boundaries**

---

## Why Does This Matter?

A classifier is a black box — you feed it features, it outputs a label. But **decision boundaries** tear open that black box and show you *where* and *why* the model draws the line between classes. They are the single most informative diagnostic tool you have for understanding what a model has learned.

If a boundary looks like a straight line on data that is clearly circular, you know immediately that your model is too simple. If a boundary perfectly memorises every training point with jagged edges, you know it is overfitting. Visualising boundaries catches these problems before they reach production.

**Spam detection angle:** Imagine your spam filter uses two features — word count and number of exclamation marks. The decision boundary is the curve (or line) that separates the "spam" region of that 2-D feature space from the "not spam" region. Every email lands somewhere in that space, and the boundary decides its fate.

## Real-World Analogy: Voting District Maps

Imagine a country where each pixel on a map is a voter. Electoral cartographers draw boundaries — gerrymandering lines — that divide the map into regions that vote Red or Blue. The boundary itself is where a voter is equally likely to go either way.

A machine learning decision boundary does exactly this:

| Voting map concept | ML concept |
|---|---|
| The map (2-D space) | Feature space (2 features as axes) |
| Each pixel / voter | A data point |
| Red vs Blue region | Class 0 vs Class 1 region |
| The boundary line | Where P(y=1 | x) = 0.5 |
| Smooth vs jagged boundary | Simple vs complex model |

**Key insight:** The boundary is where the model is *maximally uncertain*. Moving away from it in either direction, the model grows more confident.

## Plain-English Concept: How a Decision Boundary Works

For a binary classifier, the decision boundary is the set of all points **x** where:

$$P(y=1 \mid \mathbf{x}) = 0.5$$

Every point on one side has P > 0.5 → predicted class 1 (spam).  
Every point on the other side has P < 0.5 → predicted class 0 (not spam).

**For logistic regression specifically:**

$$P(y=1) = \sigma(\mathbf{w}^T \mathbf{x} + b) = 0.5 \implies \mathbf{w}^T \mathbf{x} + b = 0$$

This is a hyperplane — a straight line in 2-D, a flat plane in 3-D. That is why logistic regression can *only* learn linear boundaries. Kernels, trees, and neural networks escape this constraint.

**Visualisation recipe:**
1. Create a fine mesh grid of (x₁, x₂) points covering the feature space  
2. Feed every grid point into the trained model  
3. Colour each grid point by its predicted class (or probability)  
4. Overlay the real training points  
5. The colour boundary = the decision boundary

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D          # 3-D surface plots

from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# Reproducibility seed used throughout this notebook
SEED = 42
np.random.seed(SEED)

# Consistent colour palette — red = not-spam, blue = spam
CMAP = plt.cm.RdBu

print("Libraries loaded successfully.")

## Section 1 — Core Helper Functions

We build two reusable functions that every subsequent section will call. Understand these well — they are the engine of the entire notebook.

In [ ]:
def plot_decision_boundary(model, X, y, ax, resolution=0.02, title=""):
    """
    Shade the 2-D feature space by predicted class and overlay training points.

    Parameters
    ----------
    model      : fitted sklearn estimator with a .predict() method
    X          : (n_samples, 2) feature matrix — only 2 features!
    y          : (n_samples,) true labels
    ax         : matplotlib Axes to draw on
    resolution : step size of the mesh grid (smaller = smoother but slower)
    title      : text to display above the plot
    """
    # Expand the mesh slightly beyond the data range so no white corners appear
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    # np.arange creates a 1-D array of x values; meshgrid turns two 1-D arrays
    # into a full 2-D grid so we can evaluate the model at every (x1, x2) pair
    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, resolution),
        np.arange(y_min, y_max, resolution)
    )

    # Flatten the grid into a list of points, predict, then reshape back to grid
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # contourf fills regions between contour levels; alpha=0.4 keeps it semi-transparent
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=CMAP)

    # Overlay actual data points — edgecolors='black' makes them pop
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=CMAP,
               edgecolors='black', linewidths=0.5, s=25, zorder=3)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])


def plot_probability_contours(model, X, y, ax, levels=None, title=""):
    """
    Shade the 2-D space by predicted PROBABILITY and draw iso-probability contours.
    Requires the model to support predict_proba().
    """
    if levels is None:
        # These five levels let us see model confidence zones at a glance
        levels = [0.1, 0.3, 0.5, 0.7, 0.9]

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                          np.arange(y_min, y_max, 0.02))

    # predict_proba returns shape (n, 2); [:, 1] is the probability of class 1
    Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
    Z = Z.reshape(xx.shape)

    # Smooth background gradient showing probability continuously
    cf = ax.contourf(xx, yy, Z, levels=20, cmap='RdBu_r', alpha=0.6,
                      vmin=0, vmax=1)

    # Solid iso-probability lines at the specified levels
    cs = ax.contour(xx, yy, Z, levels=levels, colors='black',
                    linewidths=1.0, linestyles='--')
    ax.clabel(cs, inline=True, fontsize=7, fmt='%.1f')  # label each contour line

    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=CMAP,
               edgecolors='black', linewidths=0.5, s=30, zorder=3)
    ax.set_title(title, fontsize=10, fontweight='bold')
    return cf


print("Helper functions defined.")

## Section 2 — Three Synthetic Datasets (Spam Feature Spaces)

We will use three standard benchmark datasets throughout this notebook.  
Think of each as a different 2-D view of a spam feature space:

| Dataset | Spam analogy | Expected separability |
|---|---|---|
| `make_blobs` | Word count vs capital-letter ratio — cleanly separable | Linear boundary sufficient |
| `make_moons` | Two overlapping email campaigns — crescent-shaped clusters | Nonlinear boundary needed |
| `make_circles` | Concentric rings — legitimate emails surrounded by spam | Radial boundary needed |

In [ ]:
# ── Generate three 2-D datasets ───────────────────────────────────────────────
# make_blobs: two Gaussian clusters, easy for linear classifiers
X_blobs, y_blobs = make_blobs(
    n_samples=300, centers=2, cluster_std=1.2, random_state=SEED
)

# make_moons: two interleaving half-circles, hard for linear classifiers
X_moons, y_moons = make_moons(
    n_samples=300, noise=0.25, random_state=SEED
)

# make_circles: inner ring vs outer ring, the classic nonlinear test case
X_circles, y_circles = make_circles(
    n_samples=300, noise=0.1, factor=0.5, random_state=SEED
)

# Bundle them for easy iteration in plotting loops
datasets = [
    (X_blobs,   y_blobs,   "Blobs\n(linearly separable)"),
    (X_moons,   y_moons,   "Moons\n(nonlinear)"),
    (X_circles, y_circles, "Circles\n(radial)"),
]

# Quick sanity check — confirm shape and class balance
for X, y, name in datasets:
    counts = np.bincount(y)
    print(f"{name.split(chr(10))[0]:20s} → shape={X.shape}, "
          f"class counts={counts}")

## Section 3 — Five Classifiers, Three Datasets: 3×5 Boundary Grid

**What to look for:**
- **Logistic Regression** draws a straight line — excellent for Blobs, hopeless for Circles.
- **k-NN (k=5)** follows the data more closely — works well on Moons and Circles.
- **Decision Tree** creates axis-aligned rectangular steps (staircase shape).
- **Random Forest** smooths those steps slightly (ensemble averaging).
- **SVM-RBF** creates the smoothest nonlinear boundaries — often the visual ideal.

In [ ]:
# ── Define five classifiers ───────────────────────────────────────────────────
# Each is wrapped in a Pipeline with StandardScaler so features are normalised
# before training — especially important for k-NN and SVM which are distance-based.
classifiers = [
    ("Logistic\nRegression",
     Pipeline([("scaler", StandardScaler()),
                ("clf", LogisticRegression(random_state=SEED))])),

    ("k-NN\n(k=5)",
     Pipeline([("scaler", StandardScaler()),
                ("clf", KNeighborsClassifier(n_neighbors=5))])),

    ("Decision\nTree",
     Pipeline([("scaler", StandardScaler()),
                ("clf", DecisionTreeClassifier(max_depth=5, random_state=SEED))])),

    ("Random\nForest",
     Pipeline([("scaler", StandardScaler()),
                ("clf", RandomForestClassifier(n_estimators=100, random_state=SEED))])),

    ("SVM\n(RBF kernel)",
     Pipeline([("scaler", StandardScaler()),
                ("clf", SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED))])),
]

# ── Build 3 (datasets) × 5 (classifiers) subplot grid ────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
fig.suptitle("Decision Boundaries: 5 Classifiers × 3 Datasets",
             fontsize=16, fontweight='bold', y=1.01)

for row_idx, (X, y, ds_name) in enumerate(datasets):
    for col_idx, (clf_name, clf) in enumerate(classifiers):
        ax = axes[row_idx, col_idx]

        # Train classifier on the full dataset (no test split — we want to see
        # what boundary the model learned, not evaluate generalisation here)
        clf.fit(X, y)

        # Add dataset label only on first column, classifier label only on first row
        title = clf_name if row_idx == 0 else ""
        plot_decision_boundary(clf, X, y, ax, resolution=0.05, title=title)

        if col_idx == 0:
            ax.set_ylabel(ds_name, fontsize=10, fontweight='bold', labelpad=8)

plt.tight_layout()
plt.savefig("decision_boundary_grid.png", dpi=120, bbox_inches='tight')
plt.show()
print("Grid saved as decision_boundary_grid.png")

### Observations

Study the grid above before continuing:

1. **Row 1 (Blobs):** All five classifiers succeed because the data is linearly separable. The logistic regression boundary is a single clean line; the others are more complex but equally correct.
2. **Row 2 (Moons):** Logistic regression clearly fails — it draws a diagonal line through the crescent shapes and misclassifies many points. k-NN, SVM-RBF, and Random Forest adapt to the crescent geometry.
3. **Row 3 (Circles):** Logistic regression and shallow decision trees fail catastrophically. SVM-RBF and Random Forest find the circular boundary.

**Decision Tree's staircase shape** is especially visible — it can only cut parallel to the feature axes. SVM-RBF's boundary is the smoothest because the RBF kernel maps data into infinite-dimensional space where linear separation becomes easy.

## Section 4 — Boundary Complexity vs Overfitting: k-NN k-Value Sweep

**The fundamental tension in machine learning:** 
- Small k → the model memorises training data → jagged boundary → overfitting  
- Large k → the model ignores local structure → smooth boundary → underfitting

**Spam analogy:** k=1 means "classify this email the same as whichever single training email it most resembles." That is fragile — one noisy training email near the boundary misleads all nearby future emails. k=50 means "look at the 50 nearest training emails and take a majority vote," which is more robust but may miss genuine local patterns.

In [ ]:
# ── k-NN boundary evolution on make_moons ────────────────────────────────────
k_values = [1, 3, 5, 15, 30, 50]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("k-NN Decision Boundary vs k Value on make_moons\n"
             "(Red = Not-Spam region, Blue = Spam region)",
             fontsize=13, fontweight='bold')
axes_flat = axes.flatten()

for idx, k in enumerate(k_values):
    ax = axes_flat[idx]

    # Pipeline: scale first, then k-NN. This is critical — k-NN uses Euclidean
    # distance, so unscaled features with different ranges dominate the metric.
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    model.fit(X_moons, y_moons)

    # Training accuracy tells us how tightly the model fits training data
    train_acc = model.score(X_moons, y_moons)

    plot_decision_boundary(
        model, X_moons, y_moons, ax,
        resolution=0.04,
        title=f"k = {k}  |  Train acc = {train_acc:.2%}"
    )

    # Annotate with qualitative judgment
    if k == 1:
        ax.set_xlabel("⚠ Overfitting: boundary hugs every point",
                      color='darkred', fontsize=8)
    elif k == 50:
        ax.set_xlabel("⚠ Underfitting: boundary is too smooth",
                      color='navy', fontsize=8)
    else:
        ax.set_xlabel("Balanced bias-variance tradeoff", fontsize=8)

plt.tight_layout()
plt.savefig("knn_k_sweep.png", dpi=120, bbox_inches='tight')
plt.show()
print("k-sweep grid saved as knn_k_sweep.png")

## Section 5 — Polynomial Logistic Regression: Degree-Controlled Complexity

**Key idea:** Logistic regression is inherently linear. But if we *engineer* polynomial features from the originals, the decision boundary in **original** feature space becomes curved.

Adding polynomial features is like giving a flat-earth cartographer curved paper — the underlying maths stays the same, but the result can represent curved shapes.

- Degree 1 → straight line  
- Degree 2 → parabola / ellipse / hyperbola  
- Degree 5 → complex wavy curve (risk of overfitting)

In [ ]:
# ── Polynomial Logistic Regression boundary complexity ────────────────────────
degrees = [1, 2, 3, 5]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle("Polynomial Logistic Regression on make_moons\n"
             "— Degree controls boundary complexity",
             fontsize=13, fontweight='bold')

for ax, degree in zip(axes, degrees):
    # Pipeline: polynomial expansion → scale → logistic regression
    # PolynomialFeatures(degree=d) creates all combinations x1^i * x2^j
    # where i+j <= d. For degree=2 on 2 features: {1, x1, x2, x1², x1x2, x2²}
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        # C controls regularisation strength; smaller C = stronger regularisation
        ("lr", LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))
    ])
    model.fit(X_moons, y_moons)

    train_acc = model.score(X_moons, y_moons)
    n_features = model.named_steps['poly'].n_output_features_

    plot_decision_boundary(
        model, X_moons, y_moons, ax,
        resolution=0.04,
        title=(f"Degree {degree}  |  {n_features} features\n"
               f"Train acc = {train_acc:.2%}")
    )

plt.tight_layout()
plt.savefig("poly_lr_degrees.png", dpi=120, bbox_inches='tight')
plt.show()
print("Polynomial LR grid saved.")

## Section 6 — Probability Contours: Beyond Hard Boundaries

A hard decision boundary (class 0 vs class 1) discards information. **Probability contours** show the full picture:

- The **0.5 contour** is the decision boundary you already know.
- The **0.1 and 0.9 contours** show where the model is very confident.
- A **wide gap** between 0.1 and 0.9 contours means the model has a large zone of uncertainty — useful for knowing which predictions to double-check.
- A **narrow gap** means the model transitions sharply from not-spam to spam — high confidence everywhere.

**Spam analogy:** Probability contours show you not just "is this spam?" but "how sure are you?" An email sitting near the 0.5 contour should be quarantined for human review rather than auto-deleted.

In [ ]:
# ── Probability contour plot on make_blobs ────────────────────────────────────
# Logistic regression on the linearly-separable blobs is the cleanest case
# to see the S-shaped probability transition.

lr_blobs = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(C=1.0, random_state=SEED))
])
lr_blobs.fit(X_blobs, y_blobs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Hard Decision Boundary vs Probability Contours\n"
             "(Logistic Regression on make_blobs)",
             fontsize=13, fontweight='bold')

# Left: standard hard boundary
plot_decision_boundary(
    lr_blobs, X_blobs, y_blobs, axes[0],
    title="Hard Boundary (predict = 0 or 1)"
)
axes[0].set_xlabel("The boundary gives you a binary answer only", fontsize=9)

# Right: probability contours — richer story
cf = plot_probability_contours(
    lr_blobs, X_blobs, y_blobs, axes[1],
    levels=[0.1, 0.3, 0.5, 0.7, 0.9],
    title="Probability Contours (P(spam | x))"
)
plt.colorbar(cf, ax=axes[1], label='P(spam | x₁, x₂)')
axes[1].set_xlabel(
    "Gap between 0.1 and 0.9 contours = uncertainty zone", fontsize=9
)

plt.tight_layout()
plt.savefig("probability_contours.png", dpi=120, bbox_inches='tight')
plt.show()

print("Observation: The wider the gap between 0.1 and 0.9 contours,")
print("the larger the region where the model is uncertain.")
print("Points near the 0.5 contour are borderline — use with caution!")

## Section 7 — 3D Probability Surface

In 2-D feature space, probability is a scalar value at every (x₁, x₂) point. We can visualise this as a **3-D surface** where height = P(spam | x₁, x₂):

- The **S-shape** of logistic regression becomes literally visible as a tilted sigmoid surface.
- The **decision boundary** is the horizontal slice at height = 0.5.
- The **steepness** of the surface near the boundary shows how sharply the model changes its mind.

In [ ]:
# ── 3D probability surface for logistic regression ───────────────────────────
fig = plt.figure(figsize=(14, 6))
fig.suptitle("3D Probability Surface: P(spam | word_count, exclamation_marks)",
             fontsize=13, fontweight='bold')

# Create a fine grid over the blobs feature space
x_min, x_max = X_blobs[:, 0].min() - 1, X_blobs[:, 0].max() + 1
y_min, y_max = X_blobs[:, 1].min() - 1, X_blobs[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 80),
                      np.linspace(y_min, y_max, 80))

# Compute P(class=1) for every grid point
Z_prob = lr_blobs.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
Z_prob = Z_prob.reshape(xx.shape)

# Left panel: 3D surface view
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(xx, yy, Z_prob, cmap='RdBu_r', alpha=0.8,
                         rstride=2, cstride=2)  # stride controls mesh density

# Draw a horizontal plane at P=0.5 to mark the decision boundary slice
ax1.plot_surface(xx, yy, np.full_like(Z_prob, 0.5),
                  alpha=0.2, color='yellow')
ax1.set_xlabel('Feature 1 (word count)', fontsize=8)
ax1.set_ylabel('Feature 2 (exclamation marks)', fontsize=8)
ax1.set_zlabel('P(spam)', fontsize=8)
ax1.set_title('3D View\n(yellow plane = P=0.5 decision boundary)', fontsize=9)
fig.colorbar(surf, ax=ax1, shrink=0.5, aspect=10)

# Right panel: top-down view (equivalent to the 2D probability contour plot)
ax2 = fig.add_subplot(122)
cp = ax2.contourf(xx, yy, Z_prob, levels=20, cmap='RdBu_r', vmin=0, vmax=1)
ax2.contour(xx, yy, Z_prob, levels=[0.5], colors='black', linewidths=2)
ax2.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs,
             cmap=CMAP, edgecolors='black', linewidths=0.5, s=20)
ax2.set_title('Top-Down View (same surface from above)\nBlack line = decision boundary', fontsize=9)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
plt.colorbar(cp, ax=ax2, label='P(spam | x₁, x₂)')

plt.tight_layout()
plt.savefig("3d_probability_surface.png", dpi=120, bbox_inches='tight')
plt.show()
print("Key insight: The 3D surface is a tilted sigmoid (S-shape).")
print("Logistic regression learns a flat decision surface in high-dim space;")
print("the sigmoid squashes the output to [0, 1] to give probabilities.")

## Section 8 — Feature Importance and Boundary Shape

**How do irrelevant features affect the boundary?**

When we add a feature that has no real relationship to the label, a good model should (ideally) learn to ignore it. But in practice, noise features can distort the boundary and reduce generalisation. This section shows that effect by comparing models trained with 2 useful features vs 2 useful + 2 noise features.

In [ ]:
# ── Effect of irrelevant features on boundary shape ───────────────────────────
# We visualise only the first 2 features (the useful ones), but train the model
# with 2 additional random-noise columns. If the boundary changes, the model
# has been misled by noise.

rng = np.random.default_rng(SEED)

# Add 2 columns of pure random noise to moons data
noise_cols = rng.standard_normal((len(X_moons), 2))
X_moons_noisy = np.hstack([X_moons, noise_cols])

# Model A: only the 2 real features
model_clean = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(random_state=SEED))
])
model_clean.fit(X_moons, y_moons)

# Model B: 4 features (2 real + 2 noise)
model_noisy = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(random_state=SEED))
])
model_noisy.fit(X_moons_noisy, y_moons)

# ── Wrapper so plot_decision_boundary works with the 4-feature model ──────────
# We visualise boundaries in the x₁–x₂ plane with noise features fixed at zero
class PadFeaturesWrapper:
    """Adds zero-valued columns for noise features before calling the real model."""
    def __init__(self, model, total_features):
        self.model = model
        self.total_features = total_features

    def predict(self, X_2d):
        # Pad with zeros for the noise feature dimensions
        zeros = np.zeros((len(X_2d), self.total_features - 2))
        return self.model.predict(np.hstack([X_2d, zeros]))

padded_noisy = PadFeaturesWrapper(model_noisy, total_features=4)

# ── Plot side-by-side comparison ──────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Effect of Irrelevant Features on Decision Boundary",
             fontsize=13, fontweight='bold')

plot_decision_boundary(model_clean, X_moons, y_moons, ax1,
                        title="Clean: 2 real features\n"
                              f"(train acc={model_clean.score(X_moons, y_moons):.2%})")
ax1.set_xlabel("Boundary trained on meaningful features only", fontsize=9)

plot_decision_boundary(padded_noisy, X_moons, y_moons, ax2,
                        title="Noisy: 2 real + 2 random features\n"
                              f"(train acc={model_noisy.score(X_moons_noisy, y_moons):.2%})")
ax2.set_xlabel("Boundary slightly distorted by noise features", fontsize=9)

plt.tight_layout()
plt.savefig("feature_noise_effect.png", dpi=120, bbox_inches='tight')
plt.show()

# Coefficients reveal how much each feature influenced the logistic model
coefs_noisy = model_noisy.named_steps['lr'].coef_[0]
print("Logistic regression coefficients (4-feature noisy model):")
labels = ["x₁ (real)", "x₂ (real)", "noise₁", "noise₂"]
for label, coef in zip(labels, coefs_noisy):
    print(f"  {label:15s}: {coef:+.4f}")
print("\nNoise coefficients should be small — they carry no signal.")

## Section 9 — Comparing k-NN vs SVM on make_circles (Deep Dive)

One last comparison: the hardest dataset (circles) with two of our best models.
This confirms that boundary *smoothness* is not the same as boundary *quality*.
SVM-RBF is smooth because it implicitly maps to infinite dimensions;
k-NN with the right k is accurate but jagged.

In [ ]:
# ── k-NN vs SVM-RBF on make_circles ──────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X_circles, y_circles, test_size=0.3, random_state=SEED
)

models_circles = {
    "k-NN (k=5)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5))
    ]),
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED))
    ]),
    "Logistic Reg.": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(random_state=SEED))
    ]),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("make_circles: Which Boundary Generalises?",
             fontsize=13, fontweight='bold')

for ax, (name, clf) in zip(axes, models_circles.items()):
    clf.fit(X_tr, y_tr)  # train on 70%
    test_acc = clf.score(X_te, y_te)
    plot_decision_boundary(
        clf, X_circles, y_circles, ax,
        resolution=0.04,
        title=f"{name}\nTest accuracy = {test_acc:.2%}"
    )
    # Mark test points distinctly — no edgecolor black here
    ax.scatter(X_te[:, 0], X_te[:, 1], c=y_te, cmap='RdBu',
               marker='^', s=50, zorder=4,
               edgecolors='gold', linewidths=1.0,
               label='Test points')
    ax.legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.savefig("circles_comparison.png", dpi=120, bbox_inches='tight')
plt.show()
print("Gold triangles = held-out test points.")
print("Their colour (red/blue) shows their true class.")
print("Misclassified test points appear in the wrong-coloured region.")

## Section 10 — Complete Summary Table

Run this cell to print a summary of every classifier's train and test accuracy across all three datasets.

In [ ]:
# ── Comprehensive accuracy table ──────────────────────────────────────────────
print(f"{'Dataset':<12} {'Classifier':<22} {'Train Acc':>10} {'Test Acc':>10}")
print("-" * 58)

for X, y, ds_name in datasets:
    X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
        X, y, test_size=0.3, random_state=SEED
    )
    for clf_name, clf_template in classifiers:
        # Re-instantiate to get a fresh unfitted copy for a fair comparison
        import copy
        clf_copy = copy.deepcopy(clf_template)
        clf_copy.fit(X_tr_s, y_tr_s)
        tr_acc = clf_copy.score(X_tr_s, y_tr_s)
        te_acc = clf_copy.score(X_te_s, y_te_s)
        name_short = ds_name.split('\n')[0]
        clf_short  = clf_name.replace('\n', ' ')
        print(f"{name_short:<12} {clf_short:<22} {tr_acc:>10.2%} {te_acc:>10.2%}")
    print()

## Self-Check Questions

Work through these before moving to Notebook 10. Write your answers in a text cell or on paper, then reveal the solutions.

---

**Q1.** You visualize the decision boundary of k-NN (k=1) and see it perfectly separates all training points. Is this expected? Is the model good?

<details>
<summary>Show answer</summary>

**Yes, it is expected — and no, the model is probably not good.**

With k=1, every training point is its own nearest neighbour. The model assigns each point its own true label with zero error, so 100% training accuracy is guaranteed by construction. The decision boundary contorts around every single training point to achieve this.

This is the definition of **overfitting**: the model has memorised the training set rather than learned the underlying pattern. When you evaluate on unseen test data, accuracy typically drops substantially because the jagged boundary is not a good representation of the true class boundary.

A good model has training accuracy somewhat lower than 100%, with test accuracy close to training accuracy. A large gap between the two signals overfitting.

</details>

---

**Q2.** Logistic regression fails on make_circles. Without changing the algorithm, what transformation to the features would make it work?

<details>
<summary>Show answer</summary>

**Add a radial feature: r² = x₁² + x₂²**

The circles dataset has inner ring (class 1) and outer ring (class 0). The true boundary is `x₁² + x₂² = constant`. If you add the feature r² = x₁² + x₂², the data becomes **linearly separable in the new 3-D space** (original x₁, x₂, and the new r² feature). Logistic regression can then draw a flat plane that cleanly separates the two classes.

More generally:
- `PolynomialFeatures(degree=2)` automatically includes x₁², x₂², x₁x₂, which gives logistic regression enough expressive power to fit circular boundaries.
- SVM with an RBF kernel does this implicitly in infinite-dimensional space (kernel trick).

This is the power of **feature engineering**: transform the problem so that a simple algorithm can solve it.

</details>

---

**Q3.** The probability contour plot for your model shows that the 0.1 and 0.9 contours are very close to the 0.5 boundary. What does this indicate about the model's confidence?

<details>
<summary>Show answer</summary>

**The model is very sharply confident — but this may not be well-calibrated.**

When the 0.1 and 0.9 contours are close to the 0.5 boundary:
- The model transitions quickly from "almost certainly not-spam" to "almost certainly spam."
- There is a very thin zone of uncertainty.
- Points far from the boundary are assigned probabilities near 0 or 1.

This happens with **large regularisation parameter C** (weak regularisation) in logistic regression — the weights grow large, making the sigmoid's input large in magnitude, pushing outputs to extremes.

Whether this is good or bad:
- If the model is truly that certain and accuracy is high, it is fine.
- If the model is overconfident (probability=0.99 but real accuracy is 80%), it is a **calibration problem**. Use Platt scaling or isotonic regression to re-calibrate probabilities to match empirical frequencies.

Always check model calibration (reliability diagram / calibration curve) before trusting high-confidence probability outputs.

</details>